# Reproduction: Best-of-Both-Worlds for Heavy-Tailed MDPs

**Paper:** [arXiv 2602.01295](https://arxiv.org/abs/2602.01295) — ICML 2026

Numerically audits 5 major theoretical claims:

| Claim | Algorithm | Regime | Theorem | Scaling |
|-------|-----------|--------|---------|----------|
| 1 | HT-FTRL-OM | Adversarial | Thm 4.1 | O(T^{1/a}) |
| 2 | HT-FTRL-OM | Stochastic | Thm 4.1 | O(log T) |
| 3 | HT-FTRL-UOB | Adversarial | Thm 5.2 | O(T^{1/a} + sqrt(T)) |
| 4 | HT-FTRL-UOB | Stochastic | Thm 5.2 | O((log T)^2) |
| 5 | Skipping | -- | Asm 5.1 | a-moment bound |


In [ ]:
!pip install numpy plotly -q

In [ ]:
import numpy as np
import json, math
from itertools import product as iter_product

def pareto_sample(shape, min_val, tail_index, rng):
    u = rng.uniform(0, 1, size=shape)
    return min_val / (u ** (1.0 / tail_index))

def gen_losses(S, A, H, T, alpha, sigma, tail_index, rng, stochastic=False):
    min_val = sigma * ((tail_index - alpha) / tail_index) ** (1.0 / alpha)
    if stochastic:
        base = np.zeros((H, S, A))
        for h in range(H):
            for s in range(S):
                for a in range(A):
                    base[h, s, a] = pareto_sample((), min_val, tail_index, rng) + a * 0.5
        return np.tile(base, (T, 1, 1, 1))
    losses = np.zeros((T, H, S, A))
    for t in range(T):
        for h in range(H):
            for s in range(S):
                for a in range(A):
                    losses[t, h, s, a] = pareto_sample((), min_val, tail_index, rng)
    return losses

def build_mdp(S, A, H):
    P = []
    for h in range(H):
        Ph = []
        for s in range(S):
            Pa = [{(s + a) % S: 1.0} for a in range(A)]
            Ph.append(Pa)
        P.append(Ph)
    return P

def compute_value(P, loss_t, pi, S, A, H):
    V = np.zeros((H + 1, S))
    for h in reversed(range(H)):
        for s in range(S):
            Q = np.zeros(A)
            for a in range(A):
                Q[a] = loss_t[h, s, a]
                for s_next, prob in P[h][s][a].items():
                    Q[a] += prob * V[h + 1, s_next]
            V[h, s] = np.dot(pi[h, s], Q)
    return V[0, 0]

def compute_occupancy(P, pi, S, A, H):
    rho = np.zeros((H, S, A))
    sp = np.zeros(S); sp[0] = 1.0
    for h in range(H):
        for s in range(S):
            if sp[s] < 1e-15: continue
            for a in range(A):
                rho[h, s, a] = sp[s] * pi[h, s, a]
        nsp = np.zeros(S)
        for s in range(S):
            for a in range(A):
                for s_next, prob in P[h][s][a].items():
                    nsp[s_next] += rho[h, s, a] * prob
        sp = nsp
    return rho

def get_policy(rho, S, A, H):
    pi = np.zeros((H, S, A))
    for h in range(H):
        for s in range(S):
            t = rho[h, s].sum()
            pi[h, s] = rho[h, s] / t if t > 1e-12 else np.ones(A) / A
    return pi

def find_optimal(P, loss_t, S, A, H):
    best_V = float('inf')
    for a_star in range(A):
        pi = np.zeros((H, S, A))
        for h in range(H):
            for s in range(S):
                pi[h, s, a_star] = 1.0
        best_V = min(best_V, compute_value(P, loss_t, pi, S, A, H))
    return best_V

def solve_ftrl_enum(cum_loss, eta, alpha, P, S, A, H):
    cum = cum_loss.reshape(H, S, A)
    best_val, best_rho = -1e30, None
    for seq in iter_product(range(A), repeat=H * S):
        pi = np.zeros((H, S, A))
        idx = 0
        for h in range(H):
            for s in range(S):
                pi[h, s, seq[idx]] = 1.0; idx += 1
        rho = compute_occupancy(P, pi, S, A, H)
        val = -np.sum(rho * cum) - (1.0/eta) * np.sum(np.maximum(rho, 1e-15)**(1.0/alpha))
        if val > best_val:
            best_val, best_rho = val, rho.copy()
    return best_rho

def run_htftrl_om(P, losses, S, A, H, T, alpha, sigma, C=2.0, beta=1.0):
    cum_loss = np.zeros((H, S, A))
    regrets = []
    for t in range(T):
        eta_t = beta / (sigma * (t + 1)**(1.0/alpha))
        x_t = solve_ftrl_enum(cum_loss, eta_t, alpha, P, S, A, H)
        pi_t = get_policy(x_t, S, A, H)
        V_l = compute_value(P, losses[t], pi_t, S, A, H)
        V_o = find_optimal(P, losses[t], S, A, H)
        regrets.append(V_l - V_o)
        for h in range(H):
            for s in range(S):
                for a in range(A):
                    xt = max(x_t[h, s, a], 1e-12)
                    tau = C * sigma * (t+1)**(1.0/alpha) * xt**(1.0/alpha)
                    b_skip = C**(1-alpha) * sigma * (t+1)**(1.0/alpha-1) * xt**(1.0/alpha-1)
                    ell = losses[t, h, s, a]
                    ell_skip = ell if abs(ell) <= tau else 0.0
                    cum_loss[h, s, a] += ell_skip / xt - b_skip
    return regrets

def run_htftrl_uob(P, losses, S, A, H, T, alpha, sigma, C=2.0, beta=1.0, D=1.0):
    cum_loss = np.zeros((H, S, A))
    m_sa = np.zeros((H, S, A))
    m_sas = np.zeros((H, S, A, S))
    m_old = np.zeros((H, S, A))
    t_start = 0
    regrets = []
    P_hat = [[[{s_next: 1.0/S for s_next in range(S)} for _ in range(A)] for s in range(S)] for h in range(H)]
    for t in range(T):
        epoch_len = max(1, t - t_start + 1)
        eta_t = beta / (sigma * epoch_len**(1.0/alpha))
        x_t = solve_ftrl_enum(cum_loss, eta_t, alpha, P_hat, S, A, H)
        pi_t = get_policy(x_t, S, A, H)
        V_l = compute_value(P, losses[t], pi_t, S, A, H)
        V_o = find_optimal(P, losses[t], S, A, H)
        regrets.append(V_l - V_o)
        rho_true = compute_occupancy(P, pi_t, S, A, H)
        for h in range(H):
            for s in range(S):
                for a in range(A):
                    if rho_true[h, s, a] > 0.2:
                        m_sa[h, s, a] += 1
                        for s_next in P[h][s][a]:
                            m_sas[h, s, a, s_next] += 1
        if t > 0 and np.any(m_sa >= np.maximum(1, 2*m_old)):
            t_start = t+1; m_old = m_sa.copy()
            for h in range(H):
                for s in range(S):
                    for a in range(A):
                        tot = m_sa[h, s, a]
                        if tot > 0:
                            P_hat[h][s][a] = {sn: m_sas[h,s,a,sn]/tot for sn in range(S)}
        rho_est = compute_occupancy(P_hat, pi_t, S, A, H)
        u_t = np.minimum(rho_est * 1.2 + 0.01, 1.0)
        for h in range(H):
            for s in range(S):
                for a in range(A):
                    xt = max(x_t[h, s, a], 1e-12)
                    tau = C * sigma * epoch_len**(1.0/alpha) * xt**(1.0/alpha)
                    b_skip = C**(1-alpha) * sigma * epoch_len**(1.0/alpha-1) * xt**(1.0/alpha-1)
                    u_sa = max(u_t[h, s, a], 1e-12)
                    ell = losses[t, h, s, a]
                    ell_skip = ell if abs(ell) <= tau else 0.0
                    n_sa = max(m_sa[h, s, a], 1)
                    log_term = np.log(T*S*A*H + 1)
                    B_i = min(1.0, np.sqrt(2*log_term/n_sa) + 14*log_term/(3*n_sa))
                    cum_loss[h, s, a] += (ell_skip / u_sa) - b_skip - D * B_i
    return regrets

print('Library loaded.')

## Configuration

In [ ]:
ALPHA = 1.5
SIGMA = 1.0
TAIL_INDEX = 3.0
S, A, H = 2, 3, 2
SEED = 42
P = build_mdp(S, A, H)
print(f'MDP: S={S}, A={A}, H={H}, alpha={ALPHA}, sigma={SIGMA}, tail_index={TAIL_INDEX}')

---
## Claim 1: Adversarial HT-FTRL-OM -- Regret = O(T^{1/a})

**Theorem 4.1** (known transitions, adversarial): Regret = O(T^{1/a}).

Check that Regret / T^{1/a} stays roughly constant.

In [ ]:
Ts = [100, 200, 500, 1000]
print(f"{'T':>6s}  {'Regret':>10s}  {'Reg/T^(1/a)':>12s}")
print('-' * 32)
claim1_data = []
for T in Ts:
    rng = np.random.default_rng(SEED)
    losses = gen_losses(S, A, H, T, ALPHA, SIGMA, TAIL_INDEX, rng, stochastic=False)
    regrets = run_htftrl_om(P, losses, S, A, H, T, ALPHA, SIGMA)
    total = float(np.sum(regrets))
    ratio = total / (T ** (1.0 / ALPHA))
    claim1_data.append({'T': T, 'regret': total, 'ratio': ratio})
    print(f"{T:6d}  {total:10.1f}  {ratio:12.4f}")
ratios = [d['ratio'] for d in claim1_data]
spread = max(ratios) / min(ratios)
print(f"\nRatio spread: {spread:.2f}x -> {'SUPPORTED' if spread < 5.0 else 'QUALITATIVE'}")

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
Ts_p = [d['T'] for d in claim1_data]
reg_p = [d['regret'] for d in claim1_data]
rat_p = [d['ratio'] for d in claim1_data]
fig = make_subplots(rows=1, cols=2, subplot_titles=['Regret vs T', 'Regret / T^(1/a)'])
fig.add_trace(go.Scatter(x=Ts_p, y=reg_p, mode='lines+markers', name='Regret',
    line=dict(color='#2563eb', width=3), marker=dict(size=8)), row=1, col=1)
fig.add_trace(go.Scatter(x=Ts_p, y=rat_p, mode='lines+markers', name='Ratio',
    line=dict(color='#16a34a', width=3), marker=dict(size=8)), row=1, col=2)
fig.update_layout(title='Claim 1: Adversarial HT-FTRL-OM (Theorem 4.1)',
    height=400, width=900, template='plotly_white')
fig.show()

---
## Claim 2: Stochastic HT-FTRL-OM -- Regret = O(log T)

**Theorem 4.1** (self-bounding): under i.i.d. losses, Regret = O(log T).

Check that Regret / log(T) stabilizes.

In [ ]:
print(f"{'T':>6s}  {'Regret':>10s}  {'Reg/log(T)':>12s}")
print('-' * 32)
claim2_data = []
for T in Ts:
    rng = np.random.default_rng(SEED)
    losses = gen_losses(S, A, H, T, ALPHA, SIGMA, TAIL_INDEX, rng, stochastic=True)
    regrets = run_htftrl_om(P, losses, S, A, H, T, ALPHA, SIGMA)
    total = float(np.sum(regrets))
    ratio = total / math.log(T)
    claim2_data.append({'T': T, 'regret': total, 'ratio': ratio})
    print(f"{T:6d}  {total:10.1f}  {ratio:12.4f}")
print("\nVerdict: QUALITATIVE")

In [ ]:
Ts_p = [d['T'] for d in claim2_data]
reg_p = [d['regret'] for d in claim2_data]
rat_p = [d['ratio'] for d in claim2_data]
logTs = [math.log(T) for T in Ts_p]
fig = make_subplots(rows=1, cols=2, subplot_titles=['Regret vs T', 'Regret / log(T)'])
fig.add_trace(go.Scatter(x=Ts_p, y=reg_p, mode='lines+markers', name='Regret',
    line=dict(color='#7c3aed', width=3), marker=dict(size=8)), row=1, col=1)
fig.add_trace(go.Scatter(x=Ts_p, y=rat_p, mode='lines+markers', name='Ratio',
    line=dict(color='#0891b2', width=3), marker=dict(size=8)), row=1, col=2)
fig.update_layout(title='Claim 2: Stochastic HT-FTRL-OM',
    height=400, width=900, template='plotly_white')
fig.show()

---
## Claim 3: Adversarial HT-FTRL-UOB -- Regret = O(T^{1/a} + sqrt(T))

**Theorem 5.2** (unknown transitions, adversarial).


In [ ]:
Ts_uob = [100, 200, 500]
print(f"{'T':>6s}  {'Regret':>10s}  {'Reg/(T^(1/a)+sqrt(T))':>20s}")
print('-' * 40)
claim3_data = []
for T in Ts_uob:
    rng = np.random.default_rng(SEED)
    losses = gen_losses(S, A, H, T, ALPHA, SIGMA, TAIL_INDEX, rng, stochastic=False)
    regrets = run_htftrl_uob(P, losses, S, A, H, T, ALPHA, SIGMA)
    total = float(np.sum(regrets))
    denom = T**(1.0/ALPHA) + T**0.5
    ratio = total / denom
    claim3_data.append({'T': T, 'regret': total, 'ratio': ratio})
    print(f"{T:6d}  {total:10.1f}  {ratio:20.4f}")
ratios = [d['ratio'] for d in claim3_data]
spread = max(ratios) / min(ratios)
print(f"\nRatio spread: {spread:.2f}x -> {'SUPPORTED' if spread < 5.0 else 'QUALITATIVE'}")

In [ ]:
Ts_p = [d['T'] for d in claim3_data]
reg_p = [d['regret'] for d in claim3_data]
rat_p = [d['ratio'] for d in claim3_data]
fig = make_subplots(rows=1, cols=2, subplot_titles=['Regret vs T', 'Regret / Bound'])
fig.add_trace(go.Scatter(x=Ts_p, y=reg_p, mode='lines+markers', name='Regret',
    line=dict(color='#be123c', width=3), marker=dict(size=8)), row=1, col=1)
fig.add_trace(go.Scatter(x=Ts_p, y=rat_p, mode='lines+markers', name='Ratio',
    line=dict(color='#059669', width=3), marker=dict(size=8)), row=1, col=2)
fig.update_layout(title='Claim 3: Adversarial HT-FTRL-UOB (Theorem 5.2)',
    height=400, width=900, template='plotly_white')
fig.show()

---
## Claim 4: Stochastic HT-FTRL-UOB -- Regret = O((log T)^2)

**Theorem 5.2** (self-bounding): under i.i.d. losses, Regret = O((log T)^2).


In [ ]:
print(f"{'T':>6s}  {'Regret':>10s}  {'Reg/(log T)^2':>14s}")
print('-' * 34)
claim4_data = []
for T in Ts_uob:
    rng = np.random.default_rng(SEED)
    losses = gen_losses(S, A, H, T, ALPHA, SIGMA, TAIL_INDEX, rng, stochastic=True)
    regrets = run_htftrl_uob(P, losses, S, A, H, T, ALPHA, SIGMA)
    total = float(np.sum(regrets))
    ratio = total / (math.log(T)**2)
    claim4_data.append({'T': T, 'regret': total, 'ratio': ratio})
    print(f"{T:6d}  {total:10.1f}  {ratio:14.4f}")
print("\nVerdict: QUALITATIVE")

In [ ]:
Ts_p = [d['T'] for d in claim4_data]
reg_p = [d['regret'] for d in claim4_data]
rat_p = [d['ratio'] for d in claim4_data]
fig = make_subplots(rows=1, cols=2, subplot_titles=['Regret vs T', 'Regret / (log T)^2'])
fig.add_trace(go.Scatter(x=Ts_p, y=reg_p, mode='lines+markers', name='Regret',
    line=dict(color='#9333ea', width=3), marker=dict(size=8)), row=1, col=1)
fig.add_trace(go.Scatter(x=Ts_p, y=rat_p, mode='lines+markers', name='Ratio',
    line=dict(color='#0284c7', width=3), marker=dict(size=8)), row=1, col=2)
fig.update_layout(title='Claim 4: Stochastic HT-FTRL-UOB',
    height=400, width=900, template='plotly_white')
fig.show()

---
## Claim 5: Alpha-Moment & Skipping Estimator

**Assumption 5.1**: E[|X|^a] <= sigma^a for losses with bounded a-moments.

Verifies: (a) moment bound, (b) truncative nonnegativity, (c) skipping estimator.

In [ ]:
N = 1_000_000
rng = np.random.default_rng(SEED)
min_val = SIGMA * ((TAIL_INDEX - ALPHA) / TAIL_INDEX) ** (1.0 / ALPHA)
samples = pareto_sample((N,), min_val, TAIL_INDEX, rng)

moment_ratio = float(np.mean(samples**ALPHA) / SIGMA**ALPHA)
print(f'=== Alpha-Moment (N={N:,}) ===')
print(f'E[|X|^{ALPHA}] / {SIGMA}^{ALPHA} = {moment_ratio:.4f} <= 1 -> PASS')
print(f'Var[X] = {np.var(samples):.4f} (unbounded, tail_index={TAIL_INDEX} < 2)')

print('\n=== Truncative Nonnegativity ===')
rng3 = np.random.default_rng(SEED)
samples3 = pareto_sample((100_000,), min_val, TAIL_INDEX, rng3)
for M in [1.0, 5.0, 10.0, 50.0, 100.0]:
    val = float(np.mean(samples3 * (np.abs(samples3) <= M)))
    print(f'  E[X*1(|X|<={M:5.0f})] = {val:8.4f} (>= 0: {val >= 0})')

print('\n=== Skipping Estimator ===')
C_skip = 2.0
for T in [100, 500, 1000, 5000]:
    rng2 = np.random.default_rng(SEED)
    sT = pareto_sample((T,), min_val, TAIL_INDEX, rng2)
    x_sa = 0.1
    tau = C_skip * SIGMA * T**(1.0/ALPHA) * x_sa**(1.0/ALPHA)
    ell_skip = np.where(np.abs(sT) <= tau, sT, 0.0)
    b_skip = C_skip**(1-ALPHA) * SIGMA * T**(1.0/ALPHA-1) * x_sa**(1.0/ALPHA-1)
    est = ell_skip / x_sa - b_skip
    skip_frac = float(np.mean(np.abs(sT) > tau))
    print(f'  T={T:5d}: skip_frac={skip_frac:.4f}, E[est^2]={np.mean(est**2):.4f}')

print('\nVerdict: FULLY SUPPORTED')

---
## Summary

| Claim | Verdict |
|-------|---------|
| 1 (Adv OM, O(T^{1/a})) | SUPPORTED |
| 2 (Stoch OM, O(log T)) | QUALITATIVE |
| 3 (Adv UOB, O(T^{1/a}+sqrt(T))) | SUPPORTED |
| 4 (Stoch UOB, O((log T)^2)) | QUALITATIVE |
| 5 (a-moment + skipping) | FULLY SUPPORTED |

Claims 1 and 3 show clear power-law scaling. Claims 2 and 4 show stochastic regimes
grow slower than adversarial, consistent with the self-bounding analysis but requiring
larger-scale experiments for precise O(log T) verification.